# Model Inference & Streaming Decoder Verification

This notebook demonstrates how `trained_models/ctc/asl_ctc_v6.pth` runs inference on:
1. A synthetic sanity-check tensor (`(1, T, FEAT_DIM)`) to confirm model forward pass and greedy path decoding.
2. A real recorded `.npz` word take from `data/words_test` replayed frame-by-frame through `StreamingDecoder` (`src/stream_decoder.py`), matching the live webcam pipeline (`scripts/recognize_asl.py`).

In [ ]:
from pathlib import Path
import sys

# Resolve project root whether launched from repo root or inside notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root : {PROJECT_ROOT}")
print(f"src path     : {SRC_DIR} (exists={SRC_DIR.exists()})")

## 1. Load `asl_ctc_v6.pth` & Run Forward-Pass Sanity Check

In [ ]:
import numpy as np
import torch
from ctc_model import load_checkpoint, predict_paths
from seq_data import T, VOCAB, WINDOW_SEC, collapse_path
from utils import FEAT_DIM

ckpt_path = PROJECT_ROOT / "trained_models" / "ctc" / "asl_ctc_v6.pth"
ctc_model, vocab = load_checkpoint(str(ckpt_path))
ctc_model.eval()

dummy_window = np.zeros((1, T, FEAT_DIM), dtype=np.float32)
paths = predict_paths(ctc_model, dummy_window)
decoded = collapse_path(paths[0])

print(f"Input shape   : {dummy_window.shape}")
print(f"Output shape  : {paths.shape}")
print(f"Blank/zero decode: '{decoded}' (expected empty string for zero features)")

## 2. Replay Real `.npz` Word Takes Through `StreamingDecoder`
Load sample recordings from `data/words_test` (such as `LOVE`, `TEA`, `QUICK`, `WORK`) and replay them sequentially in chronological order without future lookahead.

In [ ]:
from stream_decoder import StreamingDecoder
from utils import build_features, resample_window

words_test_dir = PROJECT_ROOT / "data" / "words_test"
sample_files = sorted(words_test_dir.rglob("*.npz"))[:10]

def decode_take_file(npz_path: Path, step_sec: float = 0.25) -> str:
    data = np.load(npz_path)
    raw = data["x"] if "x" in data else data[data.files[0]]
    feats = build_features(raw)
    fps = float(data["fps"]) if "fps" in data else 30.0
    times = np.arange(len(feats)) / fps

    decoder = StreamingDecoder()
    next_t = 0.0
    for idx, t_now in enumerate(times):
        if t_now < next_t:
            continue
        next_t = t_now + step_sec
        start_idx = max(0, idx - int(WINDOW_SEC * fps))
        win = resample_window(feats[start_idx : idx + 1], T)
        path = predict_paths(ctc_model, win[None])[0]
        decoder.update(path, t_now)
    return decoder.finalize() if hasattr(decoder, "finalize") else getattr(decoder, "text", "")

for f in sample_files:
    truth = f.stem.split("_")[0]
    pred = decode_take_file(f)
    status = "MATCH" if pred == truth else "DIFF "
    print(f"[{status}] {f.name:30s} -> truth: {truth:8s} | pred: '{pred}'")

## 3. Live Webcam Execution Command
To launch real-time webcam fingerspelling recognition using `asl_ctc_v6.pth` and `trained_models/hand_landmarker.task`, run from the repository root:

```bash
python scripts/recognize_asl.py --ctc trained_models/ctc/asl_ctc_v6.pth --landmarker trained_models/hand_landmarker.task
```